Análise temporal · 1990–2025
Como o turismo bilateral evoluiu ao longo do tempo?

In [0]:
%sql
-- Como evoluiu o número anual de visitantes em cada direção do fluxo turístico?

1. Como evoluiu o número anual de visitantes em cada direção do fluxo turístico?

In [0]:
%sql
SELECT ano, origem, destino, chegadas
FROM chegadas.gold.turismo_bilateral_anual
ORDER BY ano;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * FROM chegadas.gold.turismo_bilateral_anual
WHERE destino = "Brasil" and status_dados = "incompleto"
ORDER BY ano;

In [0]:
%sql
SELECT ano, origem, destino, chegadas_observadas
FROM chegadas.gold.turismo_bilateral_anual
ORDER BY ano;

Databricks visualization. Run in Databricks to view.

**Nota metodológica:** Os dados de chegadas de visitantes
coreanos ao Brasil em 1996, 1999, 2012 e 2014 são
incompletos devido à ausência de registros para determinadas
combinações de UF e via de entrada.

Os valores apresentados nesses anos correspondem à soma
das chegadas disponíveis, não necessariamente ao total
efetivo de visitantes.

Fonte: Ministério do Turismo do Brasil e Korea Tourism
Organization.

2. Em quais anos ocorreram os maiores e menores volumes de chegadas?

In [0]:
%sql
SELECT * FROM chegadas.gold.turismo_bilateral_anual
ORDER BY chegadas_observadas DESC
LIMIT 3

In [0]:
%sql
SELECT * FROM chegadas.gold.turismo_bilateral_anual
ORDER BY chegadas_observadas
LIMIT 3;

3.  Quais períodos apresentaram crescimento, estabilidade ou retração?

In [0]:
%sql
SELECT
    ano,
    origem,
    destino,
    chegadas_observadas,

    LAG(chegadas_observadas) OVER (
        PARTITION BY origem, destino
        ORDER BY ano
    ) AS chegadas_ano_anterior,

FROM chegadas.gold.turismo_bilateral_anual
ORDER BY ano;

In [0]:
%sql
WITH dados AS (
    SELECT
    ano,
    origem,
    destino,
    chegadas_observadas,

    LAG(chegadas_observadas) OVER (
        PARTITION BY origem, destino
        ORDER BY ano
    ) AS chegadas_ano_anterior

FROM chegadas.gold.turismo_bilateral_anual
)

SELECT
    ano,
    origem,
    destino,
    chegadas_observadas,
    chegadas_ano_anterior,
    ROUND(
        ((chegadas_observadas - chegadas_ano_anterior)
        /NULLIF(chegadas_ano_anterior, 0))*100,
        2
    ) AS variacao_percentual
    

FROM dados
ORDER BY origem, destino, ano;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH dados AS (
    SELECT
        ano,
        origem,
        destino,
        chegadas_observadas,

        LAG(chegadas_observadas) OVER (
            PARTITION BY origem, destino
            ORDER BY ano
        ) AS chegadas_ano_anterior

    FROM chegadas.gold.turismo_bilateral_anual
),

variacoes AS (
    SELECT
        *,
        ROUND(
            ((chegadas_observadas - chegadas_ano_anterior)
            / NULLIF(chegadas_ano_anterior, 0)) * 100,
            2
        ) AS variacao_percentual

    FROM dados
)

SELECT
    *,
    CASE
        WHEN variacao_percentual > 0 THEN 'aumento'
        WHEN variacao_percentual < 0 THEN 'diminuição'
        WHEN variacao_percentual = 0 THEN 'estabilidade'
        ELSE "não calculável"
    END AS tendencia
FROM variacoes
ORDER BY variacao_percentual DESC;

In [0]:
%sql
WITH dados AS (
    SELECT
        ano,
        origem,
        destino,
        chegadas_observadas,

        LAG(chegadas_observadas) OVER (
            PARTITION BY origem, destino
            ORDER BY ano
        ) AS chegadas_ano_anterior

    FROM chegadas.gold.turismo_bilateral_anual
),

variacoes AS (
    SELECT
        *,
        ROUND(
            ((chegadas_observadas - chegadas_ano_anterior)
            / NULLIF(chegadas_ano_anterior, 0)) * 100,
            2
        ) AS variacao_percentual

    FROM dados
)

SELECT * FROM variacoes
ORDER BY variacao_percentual, ano
LIMIT 12;

In [0]:
%sql
WITH dados AS (
    SELECT
        ano,
        origem,
        destino,
        chegadas_observadas,

        LAG(chegadas_observadas) OVER (
            PARTITION BY origem, destino
            ORDER BY ano
        ) AS chegadas_ano_anterior

    FROM chegadas.gold.turismo_bilateral_anual
),

variacoes AS (
    SELECT
        *,
        ROUND(
            ((chegadas_observadas - chegadas_ano_anterior)
            / NULLIF(chegadas_ano_anterior, 0)) * 100,
            2
        ) AS variacao_percentual

    FROM dados
)

SELECT * FROM variacoes
ORDER BY variacao_percentual DESC
LIMIT 10;

In [0]:
%sql
WITH dados AS (
    SELECT
        ano,
        origem,
        destino,
        chegadas_observadas,

        LAG(chegadas_observadas) OVER (
            PARTITION BY origem, destino
            ORDER BY ano
        ) AS chegadas_ano_anterior

    FROM chegadas.gold.turismo_bilateral_anual
),

variacoes AS (
    SELECT
        *,
        ROUND(
            ((chegadas_observadas - chegadas_ano_anterior)
            / NULLIF(chegadas_ano_anterior, 0)) * 100,
            2
        ) AS variacao_percentual

    FROM dados
)

SELECT * FROM variacoes
ORDER BY ABS(variacao_percentual) ASC
LIMIT 12;